# Experiment 1.3c-i: Isolate lambda_max from alpha -- Track Both Independently

## Motivation and Scientific Context

**WeightWatcher** (Martin et al., 2021) characterizes trained neural network weight matrices by fitting a
power-law to the eigenvalue spectrum of **W^T W**. The fitted exponent **alpha** summarizes
spectral "health": smaller alpha indicates a flatter, more uniform eigenvalue distribution
(the hallmark of well-conditioned, generalizable layers), while larger alpha indicates
heavy-tailed concentration where a few dominant eigenvalues carry most of the spectral mass.

A critical ambiguity in the alpha metric is whether it reflects the **bulk** spectrum
(the collective distribution of all eigenvalues) or is merely **dragged** by a single outlier
eigenvalue lambda_max. If Muon optimizer's spectral control is purely about suppressing
the top singular value (sigma_1, hence lambda_max = sigma_1^2), then removing lambda_max
from the fit should equalize alpha between SGD and Muon. If instead Muon reshapes the
**entire** distribution, then alpha_clipped (excluding top eigenvalue) should still differ.

## Hypotheses

| ID | Hypothesis | Predicted Outcome |
|----|-----------|-------------------|
| H1 | Muon controls the bulk spectrum | alpha stays large/flat (low alpha) throughout training under Muon |
| H2 | SGD develops heavy tails quickly | alpha drifts toward higher values (steeper decay) under SGD |
| H3 | lambda_max grows faster under SGD | Consistent with sigma_1 growth observed in Exp 1.3b-i-A |
| H4 | Clipping lambda_max changes the story for SGD but not Muon | Because Muon already controls the outlier, removing it has minimal effect on the fit |

## Method

- **Architecture**: 4-layer deep linear network, 32x32, initialized near identity
- **Loss**: Quadratic target-matching loss: 0.5 * ||W_product @ X - W_target @ X||^2 / N
- **Optimizers**: SGD (with momentum 0.9, auto-tuned LR) vs Muon (Newton-Schulz orthogonalization, LR=0.005)
- **Spectral analysis**: At every 25 steps, compute eigenvalues of W^T W for each layer, fit power-law alpha via log-log regression of eigenvalue vs rank, and separately fit alpha_clipped excluding the top eigenvalue
- **Key metric**: |alpha - alpha_clipped| measures how much lambda_max distorts the power-law fit (i.e., how much of an outlier it is)

In [ ]:
"""
1.3c-i: Isolate lambda_max from alpha -- Track Both Independently
=================================================================

WeightWatcher alpha is the power-law exponent of the W^TW eigenvalue
distribution.  This experiment tracks lambda_max/lambda_median AND alpha
separately to determine whether Muon controls the bulk spectrum (slow
alpha drift) or just suppresses lambda_max.

HYPOTHESIS:
  - Muon controls the bulk spectrum: alpha stays large (flatter / more
    uniform eigenvalue distribution) throughout training.
  - SGD lets alpha -> 2 fast (heavy tail / concentration).
  - lambda_max grows faster under SGD (consistent with sigma_1 growth
    from 1.3b-i-A).
  - Clipping lambda_max changes the story for SGD but not Muon,
    because Muon already controls the outlier.

Power-law fit method (crude but sufficient for relative comparison):
  Sort eigenvalues descending.  Log-log linear regression of eigenvalue
  vs rank.  The negative slope is alpha.

Setup: 4-layer deep linear net, 32x32, quadratic loss, 500 steps.
       SGD (with momentum) vs Muon.
"""

In [ ]:
import numpy as np
import os

In [ ]:
np.random.seed(42)

## Section 1: Configuration

All hyperparameters are fixed here. The key design choices:

- **DIM=32**: Small enough for exact eigendecomposition at every measurement step, large enough that the 32 eigenvalues of W^T W form a meaningful spectrum for power-law fitting.
- **NUM_LAYERS=4**: Deep enough to exhibit the multiplicative singular value amplification that distinguishes Muon from SGD (product of 4 weight matrices amplifies spectral differences).
- **MEASURE_EVERY=25**: We sample the spectrum every 25 steps, giving 21 measurement points over 500 steps -- sufficient temporal resolution to track alpha drift dynamics.
- **NS_ITERS=5**: Newton-Schulz iterations for Muon's orthogonal projection. 5 iterations is standard for convergence to the polar factor U*V^T.

In [ ]:
DIM = 32
NUM_LAYERS = 4
NUM_STEPS = 500
BATCH_SIZE = 64
LR_MUON = 0.005
MOMENTUM = 0.9
NS_ITERS = 5
MEASURE_EVERY = 25

print("=== Experiment Configuration ===")
print(f"  Matrix dimension:        {DIM}x{DIM}")
print(f"  Number of layers:        {NUM_LAYERS}")
print(f"  Training steps:          {NUM_STEPS}")
print(f"  Batch size:              {BATCH_SIZE}")
print(f"  Muon LR:                 {LR_MUON}")
print(f"  Momentum (both):         {MOMENTUM}")
print(f"  Newton-Schulz iters:     {NS_ITERS}")
print(f"  Measurement interval:    every {MEASURE_EVERY} steps")
print(f"  Total measurement pts:   {NUM_STEPS // MEASURE_EVERY + 1}")
print(f"  Eigenvalues per layer:   {DIM} (from {DIM}x{DIM} W^TW)")
print(f"  Power-law fit points:    {DIM} eigenvalues per fit (log-log regression)")

In [ ]:
# Random target matrix (fixed)
W_target = np.random.randn(DIM, DIM) * 0.5

print(f"\nTarget matrix W_target: shape={W_target.shape}, scale=0.5*N(0,1)")
print(f"  Frobenius norm:  {np.linalg.norm(W_target, 'fro'):.4f}")
print(f"  Spectral norm:   {np.linalg.norm(W_target, 2):.4f}")
svs_target = np.linalg.svd(W_target, compute_uv=False)
print(f"  Singular values: max={svs_target[0]:.4f}, min={svs_target[-1]:.4f}, ratio={svs_target[0]/svs_target[-1]:.4f}")
print(f"  Condition number: {svs_target[0]/svs_target[-1]:.2f}")

In [ ]:
# Random input data (fixed batch)
X_data = np.random.randn(DIM, BATCH_SIZE) * 0.3

print(f"Input data X_data: shape={X_data.shape}, scale=0.3*N(0,1)")
print(f"  Frobenius norm:  {np.linalg.norm(X_data, 'fro'):.4f}")
print(f"  Column norms:    mean={np.mean(np.linalg.norm(X_data, axis=0)):.4f}, "
      f"std={np.std(np.linalg.norm(X_data, axis=0)):.4f}")

In [ ]:
# Output directory
SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))

In [ ]:
# Steps to print in tables
TABLE_STEPS = [0, 100, 200, 300, 500]

## Section 2: Core Functions -- Deep Linear Network

The deep linear network computes `Y = W_L @ W_{L-1} @ ... @ W_1 @ X`. Despite having no
nonlinearities, this architecture exhibits non-trivial optimization dynamics because the
loss landscape is non-convex in the individual weight matrices (even though it is convex
in the product W_product = W_L ... W_1).

**Why this matters for spectral analysis**: In a deep linear net, the singular values of
the product matrix are the products of the per-layer singular values (when layers share
singular vector bases). This means spectral imbalance in any single layer gets amplified
exponentially with depth. An optimizer that maintains spectral democracy per-layer
(like Muon) should therefore show dramatically different product-level spectral properties
compared to SGD.

**Initialization**: Each layer starts as `I + 0.1 * N(0,1)`, so initial singular values
cluster near 1.0 with small perturbations. This gives a nearly-flat initial spectrum
(low alpha) and provides a clean baseline for measuring how each optimizer reshapes the spectrum.

In [ ]:
def init_weights(num_layers, seed=42):
    """Initialize layers near identity for stability."""
    rng = np.random.RandomState(seed)
    weights = []
    for _ in range(num_layers):
        W = np.eye(DIM) + rng.randn(DIM, DIM) * 0.1
        weights.append(W.copy())
    return weights

In [ ]:
# Diagnostic: inspect initial weight spectrum
_init_w = init_weights(NUM_LAYERS)
print("Initial weight matrix diagnostics (per layer):")
for i, W in enumerate(_init_w):
    svs = np.linalg.svd(W, compute_uv=False)
    WtW_eigs = svs**2  # eigenvalues of W^T W
    alpha_init = fit_power_law_alpha(WtW_eigs) if 'fit_power_law_alpha' in dir() else "N/A"
    print(f"  Layer {i}: sigma range=[{svs[-1]:.4f}, {svs[0]:.4f}], "
          f"kappa={svs[0]/svs[-1]:.4f}, "
          f"lambda_max={WtW_eigs[0]:.4f}, lambda_median={np.median(WtW_eigs):.4f}")
print("  NOTE: alpha fitting requires fit_power_law_alpha (defined below).")
print("  Initial spectra are nearly flat (clustered around 1.0) by construction.")
del _init_w

In [ ]:
def forward(weights, X):
    """Forward pass: W_L @ ... @ W_1 @ X."""
    out = X.copy()
    for W in weights:
        out = W @ out
    return out

In [ ]:
def compute_loss(weights, X, target):
    """Loss = 0.5 * ||W_product @ X - T @ X||^2 / N."""
    pred = forward(weights, X)
    target_out = target @ X
    diff = pred - target_out
    return 0.5 * np.mean(np.sum(diff ** 2, axis=0))

In [ ]:
def compute_gradients(weights, X, target):
    """Backprop through deep linear net."""
    num_layers = len(weights)
    N = X.shape[1]

    # Forward pass storing activations
    activations = [X.copy()]
    out = X.copy()
    for W in weights:
        out = W @ out
        activations.append(out.copy())

    # Backward pass
    target_out = target @ X
    delta = (activations[-1] - target_out) / N

    grads = []
    for i in range(num_layers - 1, -1, -1):
        G = delta @ activations[i].T
        grads.insert(0, G)
        if i > 0:
            delta = weights[i].T @ delta

    return grads

In [ ]:
def newton_schulz_orthogonalize(G, num_iters=NS_ITERS):
    """
    Newton-Schulz iteration to approximate the orthogonal polar factor.
    Returns closest orthogonal matrix to G (i.e., U @ V^T from SVD).
    """
    norm = np.linalg.norm(G, ord='fro')
    if norm < 1e-12:
        return G
    X = G / norm

    for _ in range(num_iters):
        A = X.T @ X
        X = 1.5 * X - 0.5 * X @ A

    return X

## Section 3: Power-Law Alpha Fitting

### The WeightWatcher alpha metric

WeightWatcher (Martin et al., "Predicting trends in the quality of state-of-the-art neural networks
without access to training or testing data", 2021) fits a power-law to the empirical spectral
density (ESD) of **W^T W**. The exponent alpha characterizes the tail behavior:

- **alpha near 2**: Marchenko-Pastur bulk (heavy-tailed, concentrated spectrum)
- **alpha near 4+**: Well-trained, self-regularized layers (lighter tails, more uniform)
- **alpha >> 6**: Over-regularized or rank-deficient

### Our simplified fitting method

We use a crude but consistent approach: sort eigenvalues of W^T W in descending order,
compute log(eigenvalue) vs log(rank), and fit a linear regression. The negative slope
is alpha. This is a Zipf-law / rank-frequency approach rather than a maximum-likelihood
power-law fit (Clauset et al., 2009), but it suffices for **relative comparisons** between
SGD and Muon on the same architecture.

### The clipping test

`fit_power_law_alpha_clipped` excludes the **top eigenvalue** before fitting. This simulates
WeightWatcher's `xmax` clipping and directly tests whether alpha is driven by a single
outlier (lambda_max) or reflects the genuine bulk distribution shape. If |alpha - alpha_clipped|
is large, the top eigenvalue is an outlier distorting the fit.

In [ ]:
def fit_power_law_alpha(eigenvalues):
    """
    Fit power-law exponent alpha from sorted eigenvalues of W^TW.

    Method: Sort eigenvalues descending. Compute log-log linear regression
    of eigenvalue vs rank.  alpha = -slope (positive for decaying spectra).

    Returns alpha (float).
    """
    eigs = np.sort(eigenvalues)[::-1]  # descending
    eigs = eigs[eigs > 1e-30]  # remove near-zero
    n = len(eigs)
    if n < 3:
        return np.nan

    ranks = np.arange(1, n + 1).astype(float)
    log_rank = np.log(ranks)
    log_eig = np.log(eigs)

    # Linear regression: log_eig = slope * log_rank + intercept
    # alpha = -slope
    A = np.vstack([log_rank, np.ones(n)]).T
    result = np.linalg.lstsq(A, log_eig, rcond=None)
    slope = result[0][0]

    return -slope

In [ ]:
def fit_power_law_alpha_clipped(eigenvalues):
    """
    Same as fit_power_law_alpha but EXCLUDING the top eigenvalue.
    This simulates WeightWatcher's clip_xmax behavior.
    """
    eigs = np.sort(eigenvalues)[::-1]  # descending
    eigs = eigs[1:]  # exclude top eigenvalue
    eigs = eigs[eigs > 1e-30]
    n = len(eigs)
    if n < 3:
        return np.nan

    ranks = np.arange(1, n + 1).astype(float)
    log_rank = np.log(ranks)
    log_eig = np.log(eigs)

    A = np.vstack([log_rank, np.ones(n)]).T
    result = np.linalg.lstsq(A, log_eig, rcond=None)
    slope = result[0][0]

    return -slope

## Section 4: Optimizer Step Functions

### SGD with Momentum
Standard SGD with Nesterov-style momentum: `v <- momentum * v + grad`, `W <- W - lr * v`.
The learning rate is auto-tuned to the maximum stable value over 200 steps (probing from
0.05 down to 0.001). This ensures a fair comparison -- SGD gets the best LR it can handle
without diverging.

### Muon with Momentum
Muon replaces the raw gradient with its **orthogonal polar factor** via Newton-Schulz iteration:
`G_ortho = NS(grad)`, then `v <- momentum * v + G_ortho`, `W <- W - lr * v`.
The Newton-Schulz iteration computes the closest orthogonal matrix to the gradient
(equivalent to U @ V^T from the SVD of grad = U S V^T). This makes every gradient update
direction-only (unit singular values), which is the core mechanism hypothesized to control
the eigenvalue spectrum of W^T W.

**Key difference for this experiment**: SGD's update magnitude is proportional to the
gradient's singular values (amplifying already-large directions), while Muon's update has
uniform singular values (treating all directions equally). This should manifest as
different alpha trajectories -- SGD should develop heavier tails (larger alpha) as
dominant directions compound, while Muon should maintain a flatter spectrum (smaller alpha).

In [ ]:
def find_stable_lr_sgd():
    """Find maximum stable SGD learning rate."""
    candidates = [0.05, 0.03, 0.02, 0.015, 0.01, 0.007, 0.005, 0.003, 0.001]
    for lr in candidates:
        np.random.seed(42)
        weights = init_weights(NUM_LAYERS)
        velocities = [np.zeros((DIM, DIM)) for _ in range(NUM_LAYERS)]
        initial_loss = compute_loss(weights, X_data, W_target)
        stable = True
        for step in range(200):
            grads = compute_gradients(weights, X_data, W_target)
            for i in range(NUM_LAYERS):
                velocities[i] = MOMENTUM * velocities[i] + grads[i]
                weights[i] -= lr * velocities[i]
            loss = compute_loss(weights, X_data, W_target)
            if np.isnan(loss) or loss > initial_loss * 50:
                stable = False
                break
        if stable:
            return lr
    return 0.001

In [ ]:
def sgd_step(weights, velocities, lr):
    """One step of SGD with momentum."""
    grads = compute_gradients(weights, X_data, W_target)
    for i in range(len(weights)):
        velocities[i] = MOMENTUM * velocities[i] + grads[i]
        weights[i] = weights[i] - lr * velocities[i]
    return weights, velocities

In [ ]:
def muon_step(weights, velocities, lr):
    """One step of Muon with momentum."""
    grads = compute_gradients(weights, X_data, W_target)
    for i in range(len(weights)):
        ortho_grad = newton_schulz_orthogonalize(grads[i])
        velocities[i] = MOMENTUM * velocities[i] + ortho_grad
        weights[i] = weights[i] - lr * velocities[i]
    return weights, velocities

## Section 5: Measurement Engine

At each measurement step, we compute the full eigenvalue spectrum of **W^T W** for every layer.
From this spectrum we extract:

| Metric | Definition | What it tells us |
|--------|-----------|-----------------|
| **alpha** | Negative slope of log(eigenvalue) vs log(rank) | How heavy-tailed the spectrum is (higher = more concentrated) |
| **alpha_clipped** | Same fit, but excluding the top eigenvalue | Whether the bulk spectrum alone is heavy-tailed, independent of lambda_max |
| **lambda_max** | Largest eigenvalue of W^T W (= sigma_1^2) | The dominant mode's energy |
| **lambda_median** | Median eigenvalue of W^T W | Representative bulk eigenvalue |
| **outlier_ratio** | lambda_max / lambda_median | How much of a spectral outlier the top eigenvalue is |

The key diagnostic is comparing **alpha vs alpha_clipped**. If they are similar, lambda_max
is consistent with the bulk power-law. If they differ substantially, lambda_max is an outlier
that distorts the overall fit -- and the question becomes whether Muon prevents this
distortion or not.

In [ ]:
def compute_layer_spectrum_stats(W):
    """
    Given weight matrix W, compute eigenvalues of W^TW and return stats.

    Returns dict with:
      - eigenvalues: full sorted (descending) eigenvalues of W^TW
      - lambda_max, lambda_median, lambda_min
      - alpha: power-law exponent (full)
      - alpha_clipped: power-law exponent (excluding top eigenvalue)
      - outlier_ratio: lambda_max / lambda_median
    """
    WtW = W.T @ W
    eigs = np.linalg.eigvalsh(WtW)  # returns sorted ascending
    eigs = eigs[::-1]  # descending
    eigs = np.maximum(eigs, 0.0)  # ensure non-negative (numerical)

    lmax = eigs[0]
    lmin = eigs[-1]
    lmedian = np.median(eigs)
    alpha = fit_power_law_alpha(eigs)
    alpha_clipped = fit_power_law_alpha_clipped(eigs)
    outlier_ratio = lmax / lmedian if lmedian > 1e-30 else np.inf

    return {
        'eigenvalues': eigs,
        'lambda_max': lmax,
        'lambda_median': lmedian,
        'lambda_min': lmin,
        'alpha': alpha,
        'alpha_clipped': alpha_clipped,
        'outlier_ratio': outlier_ratio,
    }

In [ ]:
def run_and_measure(optimizer_name, optimizer_fn, lr, num_steps):
    """
    Run optimizer for num_steps.  At every MEASURE_EVERY steps, record:
      - alpha (per layer and mean)
      - alpha_clipped (per layer and mean)
      - lambda_max, lambda_median, lambda_min (per layer)
      - outlier_ratio = lambda_max / lambda_median (per layer and mean)
      - loss
    """
    np.random.seed(42)
    weights = init_weights(NUM_LAYERS)
    velocities = [np.zeros((DIM, DIM)) for _ in range(NUM_LAYERS)]

    # Determine measurement steps
    measure_steps = list(range(0, num_steps + 1, MEASURE_EVERY))
    if num_steps not in measure_steps:
        measure_steps.append(num_steps)
    measure_steps = sorted(set(measure_steps))

    n_measures = len(measure_steps)

    # Storage
    alpha_all = np.zeros((n_measures, NUM_LAYERS))
    alpha_clipped_all = np.zeros((n_measures, NUM_LAYERS))
    lmax_all = np.zeros((n_measures, NUM_LAYERS))
    lmedian_all = np.zeros((n_measures, NUM_LAYERS))
    lmin_all = np.zeros((n_measures, NUM_LAYERS))
    outlier_ratio_all = np.zeros((n_measures, NUM_LAYERS))
    losses = np.zeros(n_measures)

    measure_idx = 0
    diverged = False

    # Measure at step 0
    if measure_steps[0] == 0:
        for i in range(NUM_LAYERS):
            stats = compute_layer_spectrum_stats(weights[i])
            alpha_all[0, i] = stats['alpha']
            alpha_clipped_all[0, i] = stats['alpha_clipped']
            lmax_all[0, i] = stats['lambda_max']
            lmedian_all[0, i] = stats['lambda_median']
            lmin_all[0, i] = stats['lambda_min']
            outlier_ratio_all[0, i] = stats['outlier_ratio']
        losses[0] = compute_loss(weights, X_data, W_target)
        measure_idx = 1

    for step in range(1, num_steps + 1):
        weights, velocities = optimizer_fn(weights, velocities, lr)

        loss = compute_loss(weights, X_data, W_target)
        if np.isnan(loss) or loss > 1e10:
            print(f"    WARNING: {optimizer_name} diverged at step {step}!")
            # Fill remaining with NaN
            for mi in range(measure_idx, n_measures):
                alpha_all[mi] = np.nan
                alpha_clipped_all[mi] = np.nan
                lmax_all[mi] = np.nan
                lmedian_all[mi] = np.nan
                lmin_all[mi] = np.nan
                outlier_ratio_all[mi] = np.nan
                losses[mi] = np.nan
            diverged = True
            break

        if measure_idx < n_measures and step == measure_steps[measure_idx]:
            for i in range(NUM_LAYERS):
                stats = compute_layer_spectrum_stats(weights[i])
                alpha_all[measure_idx, i] = stats['alpha']
                alpha_clipped_all[measure_idx, i] = stats['alpha_clipped']
                lmax_all[measure_idx, i] = stats['lambda_max']
                lmedian_all[measure_idx, i] = stats['lambda_median']
                lmin_all[measure_idx, i] = stats['lambda_min']
                outlier_ratio_all[measure_idx, i] = stats['outlier_ratio']
            losses[measure_idx] = loss
            measure_idx += 1

    return {
        'measure_steps': np.array(measure_steps),
        'alpha': alpha_all,              # (n_measures, NUM_LAYERS)
        'alpha_clipped': alpha_clipped_all,
        'lambda_max': lmax_all,
        'lambda_median': lmedian_all,
        'lambda_min': lmin_all,
        'outlier_ratio': outlier_ratio_all,
        'losses': losses,
        'diverged': diverged,
    }

## Section 6: Run the Experiment

We now train both optimizers from identical initial conditions (same seed, same init_weights,
same data). At every 25 steps, we snapshot the full eigenvalue spectrum of W^T W for all 4
layers and compute alpha, alpha_clipped, lambda_max, lambda_median, and the outlier ratio.

This produces two parallel spectral trajectories that we can compare point-by-point.

In [ ]:
print("=" * 100)
print("1.3c-i: ISOLATE lambda_max FROM alpha -- TRACK BOTH INDEPENDENTLY")
print("=" * 100)
print(f"Setup: {NUM_LAYERS}-layer deep linear net (dim={DIM}), quadratic loss, {NUM_STEPS} steps")
print(f"Track: alpha (power-law exponent), alpha_clipped (excluding top eigenvalue),")
print(f"       lambda_max/lambda_median (outlier ratio)")
print(f"Measure every {MEASURE_EVERY} steps")
print(f"LR_Muon={LR_MUON}, Momentum={MOMENTUM}")
print("=" * 100)

In [ ]:
# Find stable SGD learning rate
lr_sgd = find_stable_lr_sgd()
print(f"\nSGD learning rate (max stable): {lr_sgd}")
print(f"Muon learning rate (fixed):     {LR_MUON}")
print(f"LR ratio (SGD/Muon):            {lr_sgd/LR_MUON:.2f}x")
print(f"\nNote: SGD LR is auto-tuned to the maximum value that does not diverge")
print(f"over 200 probe steps. This gives SGD the fairest possible comparison --")
print(f"it runs at the fastest rate it can sustain.")

In [ ]:
# Initial loss
np.random.seed(42)
w_test = init_weights(NUM_LAYERS)
loss_init = compute_loss(w_test, X_data, W_target)
print(f"Initial loss: {loss_init:.6e}")

# Inspect initial spectrum before any training
print("\nInitial spectral snapshot (step 0, before training):")
for i, W in enumerate(w_test):
    stats = compute_layer_spectrum_stats(W)
    print(f"  Layer {i}: alpha={stats['alpha']:.4f}, alpha_clipped={stats['alpha_clipped']:.4f}, "
          f"|da|={abs(stats['alpha'] - stats['alpha_clipped']):.4f}, "
          f"outlier_ratio={stats['outlier_ratio']:.4f}")
print(f"  Mean alpha:           {np.mean([compute_layer_spectrum_stats(W)['alpha'] for W in w_test]):.4f}")
print(f"  Mean outlier ratio:   {np.mean([compute_layer_spectrum_stats(W)['outlier_ratio'] for W in w_test]):.4f}")
print(f"\n  Interpretation: At initialization, alpha should be low (flat spectrum near identity)")
print(f"  and the clipping effect |alpha - alpha_clipped| should be small (no outlier yet).")

In [ ]:
# Run both optimizers
print(f"\n{'=' * 100}")
print("RUNNING OPTIMIZERS")
print("=" * 100)

In [ ]:
print("\n  Running SGD...", flush=True)
results_sgd = run_and_measure('SGD', sgd_step, lr_sgd, NUM_STEPS)
print(f"    SGD final loss: {results_sgd['losses'][-1]:.6e}")
print(f"    SGD loss reduction: {results_sgd['losses'][0]:.6e} -> {results_sgd['losses'][-1]:.6e} "
      f"({results_sgd['losses'][-1]/results_sgd['losses'][0]*100:.2f}% of initial)")
print(f"    SGD diverged: {results_sgd['diverged']}")
print(f"    SGD measurement points collected: {len(results_sgd['measure_steps'])}")

In [ ]:
print("\n  Running Muon...", flush=True)
results_muon = run_and_measure('Muon', muon_step, LR_MUON, NUM_STEPS)
print(f"    Muon final loss: {results_muon['losses'][-1]:.6e}")
print(f"    Muon loss reduction: {results_muon['losses'][0]:.6e} -> {results_muon['losses'][-1]:.6e} "
      f"({results_muon['losses'][-1]/results_muon['losses'][0]*100:.2f}% of initial)")
print(f"    Muon diverged: {results_muon['diverged']}")
print(f"    Muon measurement points collected: {len(results_muon['measure_steps'])}")

# Quick comparison of convergence
print(f"\n  Loss comparison:")
print(f"    SGD  final loss: {results_sgd['losses'][-1]:.6e}")
print(f"    Muon final loss: {results_muon['losses'][-1]:.6e}")
if results_muon['losses'][-1] > 0 and results_sgd['losses'][-1] > 0:
    ratio = results_sgd['losses'][-1] / results_muon['losses'][-1]
    print(f"    SGD/Muon loss ratio: {ratio:.4f}")
    if ratio > 1:
        print(f"    -> Muon converges to a {ratio:.1f}x lower loss than SGD")
    else:
        print(f"    -> SGD converges to a {1/ratio:.1f}x lower loss than Muon")

In [ ]:
# Get step arrays
steps_sgd = results_sgd['measure_steps']
steps_muon = results_muon['measure_steps']

## Section 7: Results Tables

We now present five tables that dissect the spectral evolution from different angles:

1. **Table 1 -- alpha(t)**: How the power-law exponent evolves over training. Tests H1 and H2.
2. **Table 2 -- Outlier ratio**: lambda_max / lambda_median over time. Tests H3.
3. **Table 3 -- Per-layer alpha**: Checks whether alpha differences are uniform across layers or layer-specific.
4. **Table 4 -- Eigenvalue summary**: Raw lambda_max, lambda_median, lambda_min values to ground the ratios.
5. **Table 5 -- Clipping effect**: |alpha - alpha_clipped| over time. The critical test for H4.

In [ ]:
def step_idx(measure_steps, step_val):
    """Return the index of step_val in measure_steps, or None."""
    idx = np.where(measure_steps == step_val)[0]
    return idx[0] if len(idx) > 0 else None

### Table 1: Power-Law Exponent alpha(t) -- Mean Across Layers

The power-law exponent alpha is computed as the negative slope of log(eigenvalue) vs log(rank)
for the eigenvalues of W^T W. This table shows how alpha evolves for both optimizers,
alongside the clipped variant (alpha_c) that excludes the top eigenvalue.

**Columns**: SGD alpha, Muon alpha, their difference, and the clipped versions.

In [ ]:
print(f"\n\n{'=' * 100}")
print("TABLE 1: POWER-LAW EXPONENT alpha(t) -- MEAN ACROSS LAYERS")
print("  alpha = negative slope of log(eigenvalue) vs log(rank)")
print("  Higher alpha => steeper decay => MORE heavy-tailed / concentrated")
print("  Lower alpha => flatter spectrum => more uniform eigenvalue distribution")
print("=" * 100)

In [ ]:
print(f"\n  {'Step':>6} | {'SGD alpha':>10} | {'Muon alpha':>11} | {'SGD-Muon':>10} | {'SGD alpha_c':>12} | {'Muon alpha_c':>13} | {'SGD-Muon clip':>14}")
print("  " + "-" * 95)

In [ ]:
for ts in TABLE_STEPS:
    idx_s = step_idx(steps_sgd, ts)
    idx_m = step_idx(steps_muon, ts)
    if idx_s is not None and idx_m is not None:
        a_sgd = np.nanmean(results_sgd['alpha'][idx_s])
        a_muon = np.nanmean(results_muon['alpha'][idx_m])
        ac_sgd = np.nanmean(results_sgd['alpha_clipped'][idx_s])
        ac_muon = np.nanmean(results_muon['alpha_clipped'][idx_m])
        print(f"  {ts:6d} | {a_sgd:10.4f} | {a_muon:11.4f} | {a_sgd - a_muon:+10.4f} | "
              f"{ac_sgd:12.4f} | {ac_muon:13.4f} | {ac_sgd - ac_muon:+14.4f}")

### Interpretation of Table 1

**What to look for**: The SGD-Muon column shows the alpha gap at each step. If this gap grows
over training, it means the two optimizers are driving the spectrum in increasingly different
directions. A positive SGD-Muon difference means SGD has higher alpha (steeper decay,
more concentrated spectrum), while Muon maintains a flatter distribution.

The alpha_clipped columns reveal whether this difference persists when we remove the top
eigenvalue. If the clipped alphas are similar between SGD and Muon, then the full-alpha
difference is driven entirely by lambda_max being an outlier under SGD. If the clipped
alphas also differ, Muon is reshaping the entire bulk spectrum, not just suppressing
the top eigenvalue.

### Table 2: Outlier Ratio -- lambda_max / lambda_median

This ratio directly measures how much of a spectral outlier the top eigenvalue is relative
to the bulk. For a perfectly flat spectrum (all eigenvalues equal), the ratio is 1.0.
For a spectrum where one eigenvalue dominates, the ratio grows large.

Under the RG/gauge-fixing interpretation, Muon's orthogonal updates should keep this ratio
bounded because the Newton-Schulz polar factor projects out scale information, preventing
any single direction from accumulating disproportionate energy.

In [ ]:
print(f"\n\n{'=' * 100}")
print("TABLE 2: OUTLIER RATIO lambda_max / lambda_median -- MEAN ACROSS LAYERS")
print("  Higher ratio => top eigenvalue is more of an outlier vs bulk")
print("=" * 100)

In [ ]:
print(f"\n  {'Step':>6} | {'SGD ratio':>10} | {'Muon ratio':>11} | {'SGD/Muon':>10}")
print("  " + "-" * 52)

In [ ]:
for ts in TABLE_STEPS:
    idx_s = step_idx(steps_sgd, ts)
    idx_m = step_idx(steps_muon, ts)
    if idx_s is not None and idx_m is not None:
        r_sgd = np.nanmean(results_sgd['outlier_ratio'][idx_s])
        r_muon = np.nanmean(results_muon['outlier_ratio'][idx_m])
        ratio_str = f"{r_sgd / r_muon:.4f}" if r_muon > 1e-10 else "N/A"
        print(f"  {ts:6d} | {r_sgd:10.4f} | {r_muon:11.4f} | {ratio_str:>10}")

### Interpretation of Table 2

**What to look for**: The SGD/Muon ratio column shows how many times larger the SGD outlier
ratio is compared to Muon. A value significantly above 1.0 means SGD develops a much more
pronounced spectral outlier. If this ratio grows over time, it confirms that SGD's
spectral concentration is a progressive phenomenon that compounds with training, while
Muon suppresses it throughout.

### Table 3: Per-Layer Alpha at Key Steps

Breaking down alpha by layer reveals whether spectral control is uniform or layer-dependent.
In a deep linear net, outer layers (closer to the loss) typically experience larger gradients,
so if Muon's spectral control is gradient-magnitude-dependent, we would see layer-specific
differences. If Muon controls all layers equally, its spectral regularization is
architecture-agnostic.

In [ ]:
print(f"\n\n{'=' * 100}")
print("TABLE 3: PER-LAYER alpha AT KEY STEPS")
print("=" * 100)

In [ ]:
print(f"\n  {'Step':>6} | ", end="")
for layer in range(NUM_LAYERS):
    print(f"{'SGD L' + str(layer):>8} {'Muon L' + str(layer):>8} | ", end="")
print()
print("  " + "-" * (8 + (18 + 3) * NUM_LAYERS))

In [ ]:
for ts in TABLE_STEPS:
    idx_s = step_idx(steps_sgd, ts)
    idx_m = step_idx(steps_muon, ts)
    if idx_s is not None and idx_m is not None:
        print(f"  {ts:6d} | ", end="")
        for layer in range(NUM_LAYERS):
            a_sgd = results_sgd['alpha'][idx_s, layer]
            a_muon = results_muon['alpha'][idx_m, layer]
            print(f"{a_sgd:8.4f} {a_muon:8.4f} | ", end="")
        print()

### Table 4: Raw Eigenvalue Summary (Layer Means)

The raw eigenvalue values ground the ratios and alpha fits in concrete numbers. 
Key things to watch:

- **lambda_max growth**: Does SGD's top eigenvalue grow much faster than Muon's? This connects to sigma_1 growth from Exp 1.3b-i-A.
- **lambda_median stability**: Does the bulk center stay stable or drift? If lambda_median grows alongside lambda_max, the outlier ratio stays bounded even if lambda_max is large.
- **lambda_min behavior**: Does the smallest eigenvalue shrink (effective rank loss) or stay bounded? Muon should maintain the bottom of the spectrum better than SGD.

In [ ]:
print(f"\n\n{'=' * 100}")
print("TABLE 4: EIGENVALUE SUMMARY (LAYER MEANS) AT KEY STEPS")
print("=" * 100)

In [ ]:
print(f"\n  {'Step':>6} | {'SGD lmax':>10} {'SGD lmed':>10} {'SGD lmin':>10} | "
      f"{'Muon lmax':>10} {'Muon lmed':>10} {'Muon lmin':>10}")
print("  " + "-" * 85)

In [ ]:
for ts in TABLE_STEPS:
    idx_s = step_idx(steps_sgd, ts)
    idx_m = step_idx(steps_muon, ts)
    if idx_s is not None and idx_m is not None:
        lmax_s = np.nanmean(results_sgd['lambda_max'][idx_s])
        lmed_s = np.nanmean(results_sgd['lambda_median'][idx_s])
        lmin_s = np.nanmean(results_sgd['lambda_min'][idx_s])
        lmax_m = np.nanmean(results_muon['lambda_max'][idx_m])
        lmed_m = np.nanmean(results_muon['lambda_median'][idx_m])
        lmin_m = np.nanmean(results_muon['lambda_min'][idx_m])
        print(f"  {ts:6d} | {lmax_s:10.4f} {lmed_s:10.4f} {lmin_s:10.4f} | "
              f"{lmax_m:10.4f} {lmed_m:10.4f} {lmin_m:10.4f}")

### Table 5: Clipping Effect -- The Critical Diagnostic

This is the most important table in the experiment. It directly answers the central question:
**Is lambda_max an outlier that distorts the power-law fit, or is it consistent with the bulk?**

- **|alpha - alpha_clipped|** measures how much removing the top eigenvalue changes the fit.
- **Large |da|** means lambda_max is a spectral outlier -- it lives in a different regime from the bulk eigenvalues.
- **Small |da|** means lambda_max is consistent with the power-law tail of the bulk.

**Prediction**: Under SGD, |da| should grow over training as lambda_max becomes an increasingly
dominant outlier. Under Muon, |da| should stay small because Muon keeps lambda_max
consistent with the bulk (or prevents it from separating in the first place).

In [ ]:
print(f"\n\n{'=' * 100}")
print("TABLE 5: CLIPPING EFFECT -- |alpha - alpha_clipped| (MEAN ACROSS LAYERS)")
print("  Large difference => lambda_max is an outlier distorting the fit")
print("  Small difference => lambda_max is consistent with bulk spectrum")
print("=" * 100)

In [ ]:
print(f"\n  {'Step':>6} | {'SGD |da|':>10} | {'Muon |da|':>11} | {'SGD-Muon':>10}")
print("  " + "-" * 52)

In [ ]:
for ts in TABLE_STEPS:
    idx_s = step_idx(steps_sgd, ts)
    idx_m = step_idx(steps_muon, ts)
    if idx_s is not None and idx_m is not None:
        da_sgd = np.nanmean(np.abs(results_sgd['alpha'][idx_s] - results_sgd['alpha_clipped'][idx_s]))
        da_muon = np.nanmean(np.abs(results_muon['alpha'][idx_m] - results_muon['alpha_clipped'][idx_m]))
        print(f"  {ts:6d} | {da_sgd:10.4f} | {da_muon:11.4f} | {da_sgd - da_muon:+10.4f}")

### Interpretation of Table 5

Compare the SGD |da| and Muon |da| columns. If SGD's clipping effect is consistently larger
and growing, it means SGD develops a spectral outlier that increasingly distorts the
power-law fit, while Muon keeps the spectrum more self-consistent. The SGD-Muon column
quantifies this difference directly.

This result has practical implications for WeightWatcher diagnostics: if lambda_max is an
outlier under SGD, then WeightWatcher's alpha metric for SGD-trained models may be
unreliable unless xmax clipping is applied. Muon-trained models would give more robust
alpha estimates regardless of clipping.

## Section 8: Formal Hypothesis Tests

We now evaluate the four hypotheses using quantitative comparisons between the final-step
spectral statistics of SGD and Muon:

| Test | Hypothesis | Metric | Expected |
|------|-----------|--------|----------|
| T1 | Alpha evolves differently | \|alpha(500) - alpha(0)\| | SGD drift > Muon drift |
| T2 | Muon keeps flatter spectrum | alpha(500) | Muon alpha < SGD alpha |
| T3 | lambda_max grows faster for SGD | lambda_max(500) / lambda_max(0) | SGD ratio > Muon ratio |
| T4 | Clipping affects SGD more | \|alpha - alpha_clipped\| at step 500 | SGD |da| > Muon |da| |

In [ ]:
print(f"\n\n{'=' * 100}")
print("KEY TESTS")
print("=" * 100)

In [ ]:
# Compute mean alpha trajectories
alpha_mean_sgd = np.nanmean(results_sgd['alpha'], axis=1)    # (n_measures,)
alpha_mean_muon = np.nanmean(results_muon['alpha'], axis=1)

print(f"\nAlpha trajectory summary (mean across {NUM_LAYERS} layers):")
print(f"  SGD  alpha: start={alpha_mean_sgd[0]:.4f}, end={alpha_mean_sgd[-1]:.4f}, "
      f"min={np.nanmin(alpha_mean_sgd):.4f}, max={np.nanmax(alpha_mean_sgd):.4f}")
print(f"  Muon alpha: start={alpha_mean_muon[0]:.4f}, end={alpha_mean_muon[-1]:.4f}, "
      f"min={np.nanmin(alpha_mean_muon):.4f}, max={np.nanmax(alpha_mean_muon):.4f}")

In [ ]:
alpha_c_mean_sgd = np.nanmean(results_sgd['alpha_clipped'], axis=1)
alpha_c_mean_muon = np.nanmean(results_muon['alpha_clipped'], axis=1)

print(f"\nAlpha_clipped trajectory summary (excluding top eigenvalue):")
print(f"  SGD  alpha_c: start={alpha_c_mean_sgd[0]:.4f}, end={alpha_c_mean_sgd[-1]:.4f}")
print(f"  Muon alpha_c: start={alpha_c_mean_muon[0]:.4f}, end={alpha_c_mean_muon[-1]:.4f}")
print(f"\nClipping effect at final step:")
print(f"  SGD  |alpha - alpha_c|: {abs(alpha_mean_sgd[-1] - alpha_c_mean_sgd[-1]):.4f}")
print(f"  Muon |alpha - alpha_c|: {abs(alpha_mean_muon[-1] - alpha_c_mean_muon[-1]):.4f}")

In [ ]:
outlier_mean_sgd = np.nanmean(results_sgd['outlier_ratio'], axis=1)
outlier_mean_muon = np.nanmean(results_muon['outlier_ratio'], axis=1)

print(f"\nOutlier ratio trajectory summary:")
print(f"  SGD  outlier_ratio: start={outlier_mean_sgd[0]:.4f}, end={outlier_mean_sgd[-1]:.4f}, "
      f"growth={outlier_mean_sgd[-1]/outlier_mean_sgd[0]:.2f}x")
print(f"  Muon outlier_ratio: start={outlier_mean_muon[0]:.4f}, end={outlier_mean_muon[-1]:.4f}, "
      f"growth={outlier_mean_muon[-1]/outlier_mean_muon[0]:.2f}x")

In [ ]:
# Use final step for tests
final_idx_s = -1
final_idx_m = -1

In [ ]:
# T1: Does alpha evolve differently for Muon vs SGD?
# Compute alpha drift: |alpha(final) - alpha(0)|
alpha_drift_sgd = abs(alpha_mean_sgd[final_idx_s] - alpha_mean_sgd[0])
alpha_drift_muon = abs(alpha_mean_muon[final_idx_m] - alpha_mean_muon[0])

In [ ]:
# Also: compute total alpha change (signed)
alpha_change_sgd = alpha_mean_sgd[final_idx_s] - alpha_mean_sgd[0]
alpha_change_muon = alpha_mean_muon[final_idx_m] - alpha_mean_muon[0]

In [ ]:
# T2: Does Muon keep a flatter spectrum (lower alpha = less heavy-tailed)?
# NOTE: higher alpha = steeper log-log slope = MORE concentrated
# Flatter spectrum = smaller alpha (eigenvalues more uniform)
alpha_final_sgd = alpha_mean_sgd[final_idx_s]
alpha_final_muon = alpha_mean_muon[final_idx_m]

In [ ]:
# T3: Does lambda_max grow faster for SGD?
lmax_growth_sgd = np.nanmean(results_sgd['lambda_max'][final_idx_s]) / np.nanmean(results_sgd['lambda_max'][0])
lmax_growth_muon = np.nanmean(results_muon['lambda_max'][final_idx_m]) / np.nanmean(results_muon['lambda_max'][0])

print(f"\nlambda_max growth analysis:")
print(f"  SGD  lambda_max: init={np.nanmean(results_sgd['lambda_max'][0]):.4f}, "
      f"final={np.nanmean(results_sgd['lambda_max'][final_idx_s]):.4f}, growth={lmax_growth_sgd:.4f}x")
print(f"  Muon lambda_max: init={np.nanmean(results_muon['lambda_max'][0]):.4f}, "
      f"final={np.nanmean(results_muon['lambda_max'][final_idx_m]):.4f}, growth={lmax_growth_muon:.4f}x")
print(f"  SGD/Muon growth ratio: {lmax_growth_sgd/lmax_growth_muon:.4f}x")

In [ ]:
# T4: Does clipping lambda_max change the story for SGD but not Muon?
clip_effect_sgd = np.nanmean(np.abs(results_sgd['alpha'][final_idx_s] - results_sgd['alpha_clipped'][final_idx_s]))
clip_effect_muon = np.nanmean(np.abs(results_muon['alpha'][final_idx_m] - results_muon['alpha_clipped'][final_idx_m]))

print(f"\nClipping effect (T4) at final step:")
print(f"  SGD  |alpha - alpha_clipped|:  {clip_effect_sgd:.4f}")
print(f"  Muon |alpha - alpha_clipped|:  {clip_effect_muon:.4f}")
print(f"  Ratio (SGD/Muon):              {clip_effect_sgd/clip_effect_muon:.4f}x" if clip_effect_muon > 1e-10 else "  Muon clipping effect near zero")
print(f"\n  This means: removing the top eigenvalue changes the power-law fit by")
print(f"  {clip_effect_sgd:.4f} for SGD vs {clip_effect_muon:.4f} for Muon.")

In [ ]:
# Print test results
test1_sgd_evolves_more = alpha_drift_sgd > alpha_drift_muon
test2_muon_flatter = alpha_final_muon < alpha_final_sgd
test3_sgd_lmax_faster = lmax_growth_sgd > lmax_growth_muon
test4_clip_sgd_more = clip_effect_sgd > clip_effect_muon

In [ ]:
print(f"""
  T1: ALPHA EVOLUTION DIFFERS BETWEEN OPTIMIZERS
      SGD  alpha drift: |alpha(500) - alpha(0)| = |{alpha_mean_sgd[final_idx_s]:.4f} - {alpha_mean_sgd[0]:.4f}| = {alpha_drift_sgd:.4f}
      Muon alpha drift: |alpha(500) - alpha(0)| = |{alpha_mean_muon[final_idx_m]:.4f} - {alpha_mean_muon[0]:.4f}| = {alpha_drift_muon:.4f}
      SGD  alpha change (signed): {alpha_change_sgd:+.4f}
      Muon alpha change (signed): {alpha_change_muon:+.4f}
      SGD drifts more: {alpha_drift_sgd:.4f} vs {alpha_drift_muon:.4f}
      -> {"CONFIRMED" if test1_sgd_evolves_more else "REJECTED"}: SGD alpha drifts {'more' if test1_sgd_evolves_more else 'less'} than Muon

  T2: MUON KEEPS FLATTER SPECTRUM (LOWER alpha = LESS HEAVY-TAILED)
      SGD  final alpha: {alpha_final_sgd:.4f}
      Muon final alpha: {alpha_final_muon:.4f}
      -> {"CONFIRMED" if test2_muon_flatter else "REJECTED"}: Muon spectrum is {'flatter' if test2_muon_flatter else 'steeper'} than SGD

  T3: lambda_max GROWS FASTER FOR SGD
      SGD  lambda_max growth factor (final/init): {lmax_growth_sgd:.4f}
      Muon lambda_max growth factor (final/init): {lmax_growth_muon:.4f}
      -> {"CONFIRMED" if test3_sgd_lmax_faster else "REJECTED"}: SGD lambda_max grows {'faster' if test3_sgd_lmax_faster else 'slower'} than Muon

  T4: CLIPPING lambda_max CHANGES STORY FOR SGD BUT NOT MUON
      SGD  |alpha - alpha_clipped| at final step:  {clip_effect_sgd:.4f}
      Muon |alpha - alpha_clipped| at final step:  {clip_effect_muon:.4f}
      -> {"CONFIRMED" if test4_clip_sgd_more else "REJECTED"}: Clipping lambda_max changes alpha {'more for SGD' if test4_clip_sgd_more else 'more for Muon'}
""")

In [ ]:
tests = [test1_sgd_evolves_more, test2_muon_flatter, test3_sgd_lmax_faster, test4_clip_sgd_more]
tests_passed = sum(tests)
tests_total = len(tests)

## Section 9: Visualization

Six panels capture the full story:

- **(a) Alpha vs Step**: The headline result -- does the power-law exponent evolve differently?
- **(b) Alpha_clipped vs Step**: Overlays full and clipped alpha to visualize the clipping effect.
- **(c) Outlier Ratio vs Step**: lambda_max / lambda_median trajectory shows outlier growth.
- **(d) lambda_max vs Step**: Raw top eigenvalue trajectory (connects to sigma_1 from 1.3b-i-A).
- **(e) Clipping Effect vs Step**: |alpha - alpha_clipped| directly visualizes how much lambda_max distorts the fit.
- **(f) Loss vs Step**: Confirms both optimizers are actually training (not just reshaping spectra of stagnant weights).

In [ ]:
print(f"\n{'=' * 100}")
print("GENERATING PLOTS")
print("=" * 100)

In [ ]:
try:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(2, 3, figsize=(20, 12))
    fig.suptitle('1.3c-i: Isolate lambda_max from alpha -- Track Both Independently\n'
                 f'{NUM_LAYERS}-layer linear net, dim={DIM}, {NUM_STEPS} steps',
                 fontsize=14, fontweight='bold')

    # --- Panel (a): alpha vs step (mean across layers) ---
    ax = axes[0, 0]
    ax.set_title('(a) Power-Law alpha vs Step (mean across layers)')
    ax.plot(steps_sgd, alpha_mean_sgd, 'b-o', linewidth=2.5, markersize=3, label='SGD')
    ax.plot(steps_muon, alpha_mean_muon, 'r--s', linewidth=2.5, markersize=3, label='Muon')
    ax.set_xlabel('Step')
    ax.set_ylabel('alpha (power-law exponent)')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

    # --- Panel (b): alpha_clipped vs step (mean across layers) ---
    ax = axes[0, 1]
    ax.set_title('(b) alpha_clipped (excl. top eigenvalue) vs Step')
    ax.plot(steps_sgd, alpha_c_mean_sgd, 'b-o', linewidth=2.5, markersize=3, label='SGD (clipped)')
    ax.plot(steps_muon, alpha_c_mean_muon, 'r--s', linewidth=2.5, markersize=3, label='Muon (clipped)')
    # Also plot unclipped for reference (thin lines)
    ax.plot(steps_sgd, alpha_mean_sgd, 'b:', linewidth=1, alpha=0.5, label='SGD (full)')
    ax.plot(steps_muon, alpha_mean_muon, 'r:', linewidth=1, alpha=0.5, label='Muon (full)')
    ax.set_xlabel('Step')
    ax.set_ylabel('alpha_clipped')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

    # --- Panel (c): lambda_max / lambda_median (outlier ratio) ---
    ax = axes[0, 2]
    ax.set_title('(c) Outlier Ratio lambda_max / lambda_median vs Step')
    ax.plot(steps_sgd, outlier_mean_sgd, 'b-o', linewidth=2.5, markersize=3, label='SGD')
    ax.plot(steps_muon, outlier_mean_muon, 'r--s', linewidth=2.5, markersize=3, label='Muon')
    ax.set_xlabel('Step')
    ax.set_ylabel('lambda_max / lambda_median')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

    # --- Panel (d): lambda_max trajectory ---
    ax = axes[1, 0]
    ax.set_title('(d) lambda_max vs Step (mean across layers)')
    ax.plot(steps_sgd, np.nanmean(results_sgd['lambda_max'], axis=1),
            'b-o', linewidth=2.5, markersize=3, label='SGD')
    ax.plot(steps_muon, np.nanmean(results_muon['lambda_max'], axis=1),
            'r--s', linewidth=2.5, markersize=3, label='Muon')
    ax.set_xlabel('Step')
    ax.set_ylabel('lambda_max (top eigenvalue of W^TW)')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

    # --- Panel (e): Clipping effect over time ---
    ax = axes[1, 1]
    ax.set_title('(e) Clipping Effect |alpha - alpha_clipped| vs Step')
    clip_diff_sgd = np.nanmean(np.abs(results_sgd['alpha'] - results_sgd['alpha_clipped']), axis=1)
    clip_diff_muon = np.nanmean(np.abs(results_muon['alpha'] - results_muon['alpha_clipped']), axis=1)
    ax.plot(steps_sgd, clip_diff_sgd, 'b-o', linewidth=2.5, markersize=3, label='SGD')
    ax.plot(steps_muon, clip_diff_muon, 'r--s', linewidth=2.5, markersize=3, label='Muon')
    ax.set_xlabel('Step')
    ax.set_ylabel('|alpha - alpha_clipped|')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

    # --- Panel (f): Loss ---
    ax = axes[1, 2]
    ax.set_title('(f) Loss vs Step')
    ax.semilogy(steps_sgd, results_sgd['losses'], 'b-o', linewidth=2.5, markersize=3, label='SGD')
    ax.semilogy(steps_muon, results_muon['losses'], 'r--s', linewidth=2.5, markersize=3, label='Muon')
    ax.set_xlabel('Step')
    ax.set_ylabel('Loss')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plot_path = os.path.join(SCRIPT_DIR, 'alpha_vs_lambda_max.png')
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"\n  Plot saved to: {plot_path}")

In [ ]:
except ImportError:
    print("\n  WARNING: matplotlib not available, skipping plots.")
    plot_path = None

## Section 10: Final Verdict and Summary

In [ ]:
print(f"\n\n{'=' * 100}")
print("FINAL VERDICT: ISOLATE lambda_max FROM alpha")
print("=" * 100)

In [ ]:
if tests_passed >= 3:
    overall = "PASS"
    detail = (
        f"  {tests_passed}/{tests_total} tests pass.\n"
        "  Muon controls the BULK spectrum (alpha stays flatter) AND suppresses\n"
        "  lambda_max growth.  Clipping lambda_max has a larger effect on SGD,\n"
        "  indicating SGD's heavy tail is driven by outlier eigenvalues while\n"
        "  Muon distributes spectral energy more uniformly."
    )
elif tests_passed >= 2:
    overall = "PARTIAL PASS"
    detail = (
        f"  {tests_passed}/{tests_total} tests pass.\n"
        f"  T1 (alpha evolves differently): {'PASS' if test1_sgd_evolves_more else 'FAIL'}\n"
        f"  T2 (Muon flatter spectrum):     {'PASS' if test2_muon_flatter else 'FAIL'}\n"
        f"  T3 (SGD lmax faster):           {'PASS' if test3_sgd_lmax_faster else 'FAIL'}\n"
        f"  T4 (clipping helps SGD more):   {'PASS' if test4_clip_sgd_more else 'FAIL'}"
    )
else:
    overall = "FAIL"
    detail = (
        f"  Only {tests_passed}/{tests_total} tests pass.\n"
        f"  T1 (alpha evolves differently): {'PASS' if test1_sgd_evolves_more else 'FAIL'}\n"
        f"  T2 (Muon flatter spectrum):     {'PASS' if test2_muon_flatter else 'FAIL'}\n"
        f"  T3 (SGD lmax faster):           {'PASS' if test3_sgd_lmax_faster else 'FAIL'}\n"
        f"  T4 (clipping helps SGD more):   {'PASS' if test4_clip_sgd_more else 'FAIL'}"
    )

In [ ]:
print(f"""
  +========================================================================+
  |  VERDICT: {overall:<63}|
  +========================================================================+
  |                                                                        |""")
for line in detail.split('\n'):
    print(f"  |  {line:<70}|")
print(f"""  |                                                                        |
  +========================================================================+
""")

In [ ]:
print("=" * 100)
print(f"  Tests passed: {tests_passed}/{tests_total}")
print(f"  Overall: {overall}")
print("=" * 100)

## Conclusions

### What this experiment establishes

This experiment decomposes the WeightWatcher alpha metric into two distinct contributions:
the **bulk spectrum shape** (measured by alpha_clipped) and the **outlier eigenvalue effect**
(measured by |alpha - alpha_clipped|). By tracking both independently under SGD and Muon,
we can determine whether Muon's spectral control operates on the bulk, the outlier, or both.

### Connection to the broader Muon-as-RG-Gauge-Fixing framework

The results from this experiment connect directly to:

- **Exp 1.3b-i-A (sigma_1 growth rate)**: If lambda_max = sigma_1^2 grows faster under SGD,
  this confirms the sigma_1 feedback loop identified in that experiment. The alpha decomposition
  here tells us whether that sigma_1 growth is an isolated outlier or part of a broader
  spectral concentration.
  
- **Exp 1.3a-i (effective rank)**: If Muon maintains a flatter spectrum (lower alpha), this
  is consistent with higher effective rank. The alpha metric gives us a complementary view
  of the same phenomenon through the lens of heavy-tail theory.

- **WeightWatcher theory (Martin et al.)**: The clipping test directly validates whether
  WeightWatcher's alpha is a reliable diagnostic for Muon-trained models vs SGD-trained models.
  If Muon keeps lambda_max consistent with the bulk, WeightWatcher alpha is more trustworthy
  for Muon models.

### Implications for practice

If the hypothesis holds (Muon controls bulk + outlier), then:
1. Muon-trained models should have more reliable WeightWatcher diagnostics
2. The alpha metric under Muon reflects genuine spectral health, not just outlier suppression
3. SGD models may need xmax clipping for meaningful alpha estimates
4. The orthogonal gradient projection in Muon is not just suppressing sigma_1 -- it is
   reshaping the entire spectral distribution toward greater democracy